In [1]:
import netCDF4 as nc

In [2]:
nc_file = nc.Dataset('mesh_data.nc', 'r')

In [10]:
nc_file.dimensions['frame'].size

100

In [4]:
nc_file['HelfrichEnergy'][:]

masked_array(data=[20363.25566431,   165.99085286,   122.88198833,
                     120.63813875,   119.15457191,   118.02824652,
                     117.52505918,   117.18541159,   116.94040491,
                     116.78264737,   116.32695975,   116.00848209,
                     115.79566578,   115.9917371 ,   116.09218307,
                     116.36539774,   116.36384643,   116.25625892,
                     116.30202455,   116.51310307,   116.43484393,
                     116.31427175,   116.11731093,   116.02014622,
                     115.89159456,   116.11351071,   117.46833949,
                     116.89918166,   116.73471091,   116.84487201,
                     118.39169037,   118.55758726,   118.92745988,
                     121.87766688,   120.10403875,   120.45727912,
                     119.65797282,   119.65968458,   119.57757622,
                     118.88476429,   119.33374182,   119.22064445,
                     118.8473691 ,   118.9500067 ,   118.94365

## Read Mesh Trajectory to VTK files:

In [5]:
import vtk
import numpy as np


for i in range(nc_file.dimensions['frame'].size):
    # Assuming you have vertices and connections as arrays
    mask = nc_file['vertices'][i].mask
    data = nc_file['vertices'][i].data
    vertices = data[~mask].reshape(-1,3)
    mask = nc_file['faces'][i].mask
    data = nc_file['faces'][i].data
    faces = data[~mask].reshape(-1,3)

    vertices_array = np.array(vertices)
    connections_array = np.array(faces)
    
    # Create a vtkPoints object
    vertices = vtk.vtkPoints()
    for vertex in vertices_array:
        vertices.InsertNextPoint(vertex)
    
    # Create a vtkCellArray object
    connections = vtk.vtkCellArray()
    for connection in connections_array:
        # Ensure connection is a 1D array
        if isinstance(connection, np.ndarray):
            connection = connection.tolist()
        polygon = vtk.vtkPolygon()
        for idx in connection:
            polygon.GetPointIds().InsertNextId(idx)
        connections.InsertNextCell(polygon)
    
    # Create a vtkPolyData object
    polydata = vtk.vtkPolyData()
    polydata.SetPoints(vertices)
    polydata.SetPolys(connections)
    
    # Write the PolyData to a VTK file
    writer = vtk.vtkPolyDataWriter()
    writer.SetFileName("output_mesh"+str(i)+".vtk")
    writer.SetInputData(polydata)
    writer.Write()

In [ ]:


mask = nc_file['vertices'][nc_file.dimensions['frame'].size-1].mask
data = nc_file['vertices'][nc_file.dimensions['frame'].size-1].data
vertices = data[~mask].reshape(-1,3)
mask = nc_file['faces'][nc_file.dimensions['frame'].size-1].mask
data = nc_file['faces'][nc_file.dimensions['frame'].size-1].data
faces = data[~mask].reshape(-1,3)

# Write to .obj file
obj_file = "mesh_data.obj"
with open(obj_file, "w") as f:
    # Write vertices
    for vertex in vertices:
        f.write("v {} {} {}\n".format(vertex[0], vertex[1], vertex[2]))

    # Write faces
    for face in faces:
        f.write("f {} {} {}\n".format(face[0]+1, face[1]+1, face[2]+1))

print("Mesh data has been written to", obj_file)
